<a href="https://colab.research.google.com/github/Muhammad-Wasil-Ali/Rag-using-Langchain/blob/main/Rag_Using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install  google-genai langchain tiktoken rapidocr-onnxruntime openrouter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.4/748.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 64.7 MB/s eta 0:00:00


In [3]:
from google.colab import userdata

GEMINI_API_KEY=userdata.get("GEMINI_API_KEY")
OPENOUTER_API_KEY=userdata.get("OPENOUTER_API_KEY")

Data Ingestion

Data Retreiver

Data Generation

In [4]:
!pip install langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
import requests
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_691/3145855899.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [15]:
loader=TextLoader("/content/state_of_the_union.txt")

In [18]:
document=loader.load()

In [22]:
document[0].page_content[:200]

'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. '

Chunking Data

In [23]:
!pip install -U langchain-text-splitters

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
text_split=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [47]:
text=text_split.split_documents(document)

'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny.'

In [31]:
!pip install -q langchain-google-genai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 103.2 MB/s eta 0:00:00


In [48]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [49]:
embeddings=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",api_key=GEMINI_API_KEY)

In [50]:
vectorstore=FAISS.from_documents(text,embeddings)

In [42]:
query = "What is this document about?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
Up to eight state-of-the-art factories in one place. 10,000 new good-paying jobs. 

Some of the most sophisticated manufacturing in the world to make computer chips the size of a fingertip that power the world and our everyday lives. 

Smartphones. The Internet. Technology we have yet to invent. 

But that’s just the beginning. 

Intel’s CEO, Pat Gelsinger, who is here tonight, told me they are ready to increase their investment from  
$20 billion to $100 billion.

--- Chunk 2 ---
The U.S. Department of Justice is assembling a dedicated task force to go after the crimes of Russian oligarchs.  

We are joining with our European allies to find and seize your yachts your luxury apartments your private jets. We are coming for your ill-begotten gains.

--- Chunk 3 ---
And we will, as one people. 

One America. 

The United States of America. 

May God bless you all. May God protect our troops.


In [51]:
!pip install  langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 11.1 MB/s eta 0:00:00


In [52]:
retriever=vectorstore.as_retriever()

In [79]:
from langchain_core.prompts import ChatPromptTemplate

template=template = """You are a helpful assistant.
Use ONLY the given context to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""



In [80]:
prompt=ChatPromptTemplate.from_template(template)

In [81]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [82]:
llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    openai_api_key=OPENOUTER_API_KEY,  # poori key yahan paste karo
    openai_api_base="https://openrouter.ai/api/v1"
)

In [83]:
output_parser=StrOutputParser()

In [84]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [86]:
response=rag_chain.invoke("Why all members together again and when?")

In [87]:
response

'All members are together again because COVID‑19 kept them apart the previous year, and they have reunited this year.'